# Flight Booking Web Scraping Project

## Domain Definition & Product Idea

**Project Title:** Flight Price Analysis & Affordability Tracking System

**Domain:** Travel & Tourism Industry - Flight Booking Data Analysis

**Product Idea:** 
Build a dataset of flight prices from multiple origin-destination routes and dates. This enables:
- Price trend analysis across routes
- Budget level classification (Low/Medium/High)
- Affordability scoring for travelers
- Route recommendation based on budget constraints

**Data Source:** Flight prices from booking.kayak.com

**Target Users:** Budget-conscious travelers, travel planners, data analysts

**Business Value:** Help users identify cheapest destinations and best travel timing

## 2. Web Scraping & Crawling Implementation

Collecting flight data from multiple routes using realistic synthetic generation.

---


In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, StaleElementReferenceException
)
import time
import re
import pandas as pd
import random

# ── Browser setup ──────────────────────────────────────────────────────────────
chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationDetection")
chrome_options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)
driver = webdriver.Chrome(options=chrome_options)
wait   = WebDriverWait(driver, 20)

routes = [("CAI", "ICN"), ("ICN", "CAI"),
          ("CAI", "PAR"), ("PAR", "CAI")
          ]
dates  = ["2026-05-20", "2026-05-25"]
data   = []


# ── Helpers ────────────────────────────────────────────────────────────────────
def safe_text(element, css_selector, default=None):
    try:
        return element.find_element(By.CSS_SELECTOR, css_selector).text.strip() or default
    except NoSuchElementException:
        return default


def parse_price(text):
    """Extract a number from any price string: '$223', '8,191£', etc."""
    if not text:
        return None
    match = re.search(r'[\d,]+\.?\d*', text)
    if not match:
        return None
    try:
        return float(match.group().replace(",", ""))
    except ValueError:
        return None


def direct_text_node(element):
    """
    Return the first non-empty direct text node of an element via JS.
    Needed when the price is a bare text node sitting AFTER a child element
    (e.g. a tooltip div), which Selenium's .text concatenates with child text.
    """
    return driver.execute_script("""
        var el = arguments[0];
        for (var node of el.childNodes) {
            if (node.nodeType === 3 && node.textContent.trim()) {
                return node.textContent.trim();
            }
        }
        return null;
    """, element)


def get_real_flight_cards():
    all_cards = driver.find_elements(By.CSS_SELECTOR, ".Fxw9-result-item-container")
    real_cards = [
        c for c in all_cards
        if c.find_elements(By.CSS_SELECTOR, ".e2GB-price-text")
        and c.is_displayed()
        and c.size['height'] > 0
    ]
    return real_cards


# ── Fare price via JS direct text node ────────────────────────────────────────
def get_fare_price(card):
    # Multi-fare layout: price is direct text node in .rE1p-multi-header
    try:
        container = card.find_element(
            By.CSS_SELECTOR, ".rE1p-mod-title-default .rE1p-multi-header")
        price = direct_text_node(container)
        if price:
            return price
    except NoSuchElementException:
        pass

    # Single-fare layout: cabin + price concatenated in .rE1p-mod-title-default
    try:
        container = card.find_element(By.CSS_SELECTOR, ".rE1p-mod-title-default")
        full_text = container.text.strip()
        match = re.search(r'[\$£€][\d,]+|[\d,]+[\$£€]', full_text)
        if match:
            return match.group()
    except NoSuchElementException:
        pass

    return None


# ── Provider price via JS direct text node ────────────────────────────────────
def get_provider_price(row):
    try:
        container = row.find_element(By.CSS_SELECTOR, ".c7v8g-price-top-container")
        return direct_text_node(container)
    except NoSuchElementException:
        return None


# ── Scrape fare cards (Economy, Fully Flex, Business …) ───────────────────────
# ── Scrape fare cards (Economy, Fully Flex, Business …) ───────────────────────
def scrape_fare_cards():
    result = {}
    idx = 0

    def get_visible_fare_cards():
        try:
            modal_el = driver.find_element(
                By.CSS_SELECTOR, ".lzu8-modal-details-fare-group-content")
            cards = modal_el.find_elements(
                By.CSS_SELECTOR, ".lzu8.lzu8-mod-modal-details-card")
            if cards:
                return cards
        except NoSuchElementException:
            pass
        cards = driver.find_elements(By.CSS_SELECTOR, ".lzu8.lzu8-mod-modal-details-card")
        if cards:
            return cards
        cards = driver.find_elements(By.CSS_SELECTOR, ".c-5pd-list")
        if cards:
            return cards
        cards = [
            el for el in driver.find_elements(By.CSS_SELECTOR,
                "[role='dialog'] [role='group']")
            if el.find_elements(By.CSS_SELECTOR, ".rE1p-multi-header, .rE1p-top-row")
        ]
        if cards:
            return cards
        dialogs = driver.find_elements(By.CSS_SELECTOR, "[role='dialog']")
        return [d for d in dialogs if d.find_elements(
            By.CSS_SELECTOR, ".rE1p-multi-header, [class*='fare-price']")]

    def get_cabin_name(card):
        """Try every known selector for cabin name across all fare tiers."""
        for sel in [
            ".WgGj",                    # Economy tiers
            ".rE1p-title",              # some layouts
            "[class*='cabinName']",
            "[class*='cabin-name']",
            "[class*='CabinName']",
            ".Hy6H",
            "[class*='fareTitle']",
            "[class*='fare-title']",
            "[class*='FareTitle']",
            "[class*='fareType']",
            "[class*='fare-type']",
            "h3", "h4",                 # fallback headings
        ]:
            val = safe_text(card, sel)
            if val:
                return val
        # Last resort: grab first short text line from the card
        try:
            full = card.text.strip().split("\n")[0]
            if full and len(full) < 40:
                return full
        except Exception:
            pass
        return None

    # Wait for fare cards to appear
    for _ in range(10):
        if get_visible_fare_cards():
            break
        time.sleep(0.8)

    # Get carousel dot buttons — one per fare, always rendered even when hidden
    dots = []
    for dot_sel in [
        ".c-5pd-inner [role='group']",
        ".c-5pd-inner [role='button']",
        ".c-5pd-inner li",
        "[class*='carousel'] [role='group']",
        "[class*='carousel'] [role='button']",
    ]:
        dots = driver.find_elements(By.CSS_SELECTOR, dot_sel)
        if dots:
            break

    total_fare_count = len(dots) if dots else len(get_visible_fare_cards())
    print(f"      🎫 Found {total_fare_count} fare card(s)")

    def scrape_current_page():
        """Scrape only the currently ACTIVE/VISIBLE cards on this carousel page."""
        nonlocal idx
        if idx >= 3:
            return
        # Only grab cards that are actually visible/active right now
        all_cards = get_visible_fare_cards()
        active_cards = []
        for card in all_cards:
            try:
                # Skip cards that are hidden (off-screen via carousel transform)
                style = card.get_attribute("style") or ""
                aria_hidden = card.get_attribute("aria-hidden") or ""
                if "aria-hidden" in aria_hidden and aria_hidden == "true":
                    continue
                # Check if it has any visible text content at all
                if card.text.strip():
                    active_cards.append(card)
            except Exception:
                continue

        # If filtering removed everything, fall back to all cards
        if not active_cards:
            active_cards = all_cards

        for card in active_cards:
            try:
                price_raw = get_fare_price(card) or safe_text(card, ".e2GB-price-text")
                price_val = parse_price(price_raw)
                cabin     = get_cabin_name(card)

                if not price_val:
                    continue  # skip ghost/empty cards entirely

                idx += 1
                p = f"fare_{idx}_"
                amenities = [
                    el.text.strip()
                    for el in card.find_elements(By.CSS_SELECTOR, ".hk_J-message-container")
                    if el.text.strip()
                ]
                result[p + "cabin"]          = cabin
                result[p + "price"]          = price_val
                result[p + "carry_on"]       = next((t for t in amenities if "carry"   in t.lower()), None)
                result[p + "checked_bag"]    = next((t for t in amenities if "checked" in t.lower()), None)
                result[p + "seat_selection"] = next((t for t in amenities if "seat"    in t.lower()), None)
                print(f"        Fare {idx}: {cabin} → {price_raw}")
            except Exception as e:
                print(f"        ⚠️  Fare card error: {e}")

    if dots:
        # Click each dot to jump directly to that page, then scrape
        for dot_num, dot in enumerate(dots):
            try:
                driver.execute_script("arguments[0].click();", dot)
                time.sleep(1.2)  # wait for carousel animation
                scrape_current_page()
                if idx >= total_fare_count:
                    break
            except Exception as e:
                print(f"        ⚠️  Dot {dot_num} click error: {e}")
    else:
        # No dots found — just scrape what's visible (fallback)
        scrape_current_page()

    return result


# ── Scrape booking providers (BudgetAir, Kiwi, eDreams …) ────────────────────
def scrape_providers():
    result = {}

    rows = driver.find_elements(By.CSS_SELECTOR, ".RBZl-row-wrapper.RBZl-mod-top-row")
    print(f"      🏪 Found {len(rows)} provider row(s) (raw)")

    if not rows:
        print("      ⚠️  No provider rows found.")
        return result

    valid_idx = 0
    for row in rows:
        try:
            name       = safe_text(row, ".veIp-provider-name")
            price_text = get_provider_price(row)

            if not name or not price_text:
                print(f"        ⏭️  Skipping non-provider row (name={name}, price={price_text})")
                continue

            valid_idx += 1
            p = f"provider_{valid_idx}_"
            result[p + "name"]  = name
            result[p + "price"] = parse_price(price_text)
            print(f"        Provider {valid_idx}: {name} → {price_text}")

        except Exception as e:
            print(f"        ⚠️  Provider row error: {e}")

    print(f"      🏪 Kept {valid_idx} valid provider(s)")
    return result


# ── Main loop ──────────────────────────────────────────────────────────────────
try:
    for origin, dest in routes:
        for date in dates:
            search_url = (
                f"https://booking.kayak.com/flights/{origin}-{dest}/{date}"
                "?sort=bestflight_a"
            )
            print(f"\n🔍 Scraping: {search_url}")
            driver.get(search_url)

            try:
                wait.until(EC.presence_of_element_located(
                    (By.CSS_SELECTOR, ".Fxw9-result-item-container")))
            except TimeoutException:
                print(f"   ⚠️  No results for {origin}→{dest} on {date}, skipping.")
                continue

            time.sleep(8)

            # ── Phase 1: populate flight_cards_data FIRST ──────────────────
            flight_cards_data = []
            cards = get_real_flight_cards()
            cards = cards[:5]  # cap at 5
            print(f"📦 Found {len(cards)} real flight card(s) (capped at 5)")

            # ✅ FIX: append loop runs BEFORE dedup
            for i, flight in enumerate(cards):
                try:
                    time_spans = flight.find_elements(
                        By.CSS_SELECTOR, ".vmXl.vmXl-mod-variant-large span")
                    flight_cards_data.append({
                        "card_index":  i,
                        "origin":      origin,
                        "destination": dest,
                        "date":        date,
                        "airline":     safe_text(flight, ".c_cgF[dir='ltr']"),
                        "from_time":   time_spans[0].text if len(time_spans) > 0 else None,
                        "to_time":     time_spans[2].text if len(time_spans) > 2 else None,
                        "stops":       safe_text(flight, ".vmXl.vmXl-mod-variant-default span"),
                        "duration":    safe_text(flight, ".xdW8 .vmXl"),
                        "cabin_class": safe_text(flight, ".Hy6H"),
                        "base_price":  parse_price(safe_text(flight, ".e2GB-price-text")),
                    })
                except Exception as e:
                    print(f"   ⚠️  Skipped card {i}: {e}")

            # ✅ FIX: dedup runs AFTER flight_cards_data is populated
            seen = set()
            unique_cards = []
            for card in flight_cards_data:
                key = (card['airline'], card['from_time'], card['to_time'], card['base_price'])
                if key not in seen:
                    seen.add(key)
                    unique_cards.append(card)
            flight_cards_data = unique_cards
            print(f"📦 After dedup: {len(flight_cards_data)} unique flights")

            # ── Phase 2: click each card and scrape fare + provider detail ──
            for card_info in flight_cards_data:
                i = card_info["card_index"]
                print(f"\n  ✈️  Card {i}: {card_info['airline']} | "
                      f"{card_info['from_time']}→{card_info['to_time']} | "
                      f"${card_info['base_price']}")

                fare_data     = {}
                provider_data = {}

                try:
                    driver.get(search_url)
                    wait.until(EC.presence_of_element_located(
                        (By.CSS_SELECTOR, ".Fxw9-result-item-container")))
                    time.sleep(6)

                    fresh_cards = get_real_flight_cards()
                    if i >= len(fresh_cards):
                        print(f"      ⚠️  Card {i} not present after reload.")
                        data.append({**card_info})
                        continue

                    target_card = fresh_cards[i]
                    select_btn  = None

                    for selector in [
                        ".e2GB-price-text",
                        "button[class*='select']", "a[class*='select']",
                        "[class*='booking-btn']",  "[class*='book-btn']",
                        "[class*='BFM']",          "button[class*='price']",
                        "[data-testid*='select']",
                        "[aria-label*='Select']",  "[aria-label*='select']",
                    ]:
                        try:
                            select_btn = target_card.find_element(By.CSS_SELECTOR, selector)
                            break
                        except NoSuchElementException:
                            continue

                    if select_btn is None:
                        for btn in target_card.find_elements(By.TAG_NAME, "button"):
                            if "select" in btn.text.lower():
                                select_btn = btn
                                break

                    if select_btn:
                        original_window = driver.current_window_handle
                        before_handles  = set(driver.window_handles)

                        driver.execute_script(
                            "arguments[0].scrollIntoView({block:'center'});", select_btn)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", select_btn)
                        time.sleep(4)

                        new_handles = set(driver.window_handles) - before_handles
                        if new_handles:
                            driver.switch_to.window(new_handles.pop())
                            print(f"      🪟 New tab: {driver.current_url}")
                        else:
                            print(f"      🔗 Same tab: {driver.current_url}")

                        time.sleep(3)

                        fare_data     = scrape_fare_cards()
                        provider_data = scrape_providers()

                        if new_handles:
                            driver.close()
                            driver.switch_to.window(original_window)
                        else:
                            driver.back()
                            time.sleep(4)
                    else:
                        print(f"      ⚠️  No select button on card {i}.")

                except StaleElementReferenceException:
                    print(f"      ⚠️  Stale element on card {i}.")
                except Exception as e:
                    print(f"      ⚠️  Error on card {i}: {e}")

                row = {**card_info, **fare_data, **provider_data}
                row.pop("card_index", None)
                data.append(row)
                time.sleep(random.uniform(2, 4))

finally:
    driver.quit()


# ── Save dataset ───────────────────────────────────────────────────────────────
if data:
    df = pd.DataFrame(data)

    base_cols = [
        "origin", "destination", "date",
        "airline", "from_time", "to_time", "duration", "stops",
        "cabin_class", "base_price",
    ]

    fare_cols = sorted(
        [c for c in df.columns if c.startswith("fare_")],
        key=lambda x: (int(x.split("_")[1]), x)
    )
    provider_cols = sorted(
        [c for c in df.columns if c.startswith("provider_")],
        key=lambda x: (int(x.split("_")[1]), x)
    )

    all_cols = [c for c in base_cols + fare_cols + provider_cols if c in df.columns]
    df = df[all_cols]

    df.to_csv("flights_dataset.csv", index=False)
    df.to_excel("flights_dataset.xlsx", index=False)
    df.to_json("flights_dataset.json", orient="records", indent=2)

    print(f"\n✅ Saved {len(df)} record(s).")
    print(f"   Columns ({len(df.columns)}): {list(df.columns)}")
    print(df.to_string())
else:
    print("\n⚠️  No data collected.")


🔍 Scraping: https://booking.kayak.com/flights/CAI-ICN/2026-05-20?sort=bestflight_a
📦 Found 5 real flight card(s) (capped at 5)
📦 After dedup: 5 unique flights

  ✈️  Card 0: Emirates | 8:05 pm→5:00 pm
+1 | $594.0
      🔗 Same tab: https://booking.kayak.com/flights/CAI-ICN/2026-05-20?sort=bestflight_a#dialog
      🎫 Found 17 fare card(s)
        Fare 1: Eco Saver → $594
        Fare 2: Eco Flex → $644
        Fare 3: Eco Flex Plus → $1,091
      🏪 Found 24 provider row(s) (raw)
        Provider 1: Expedia → $594
        Provider 2: Expedia → $594
        Provider 3: Booking.com → $597
        Provider 4: Kiwi.com → $597
        Provider 5: eDreams → $598
        Provider 6: Priceline → $604
        Provider 7: OneTravel → $604
        Provider 8: CheapOair → $604
        Provider 9: Mytrip → $611
        Provider 10: Gotogate → $613
        ⏭️  Skipping non-provider row (name=None, price=$644)
        ⏭️  Skipping non-provider row (name=None, price=$644)
        ⏭️  Skipping non-provid


## 3. Robots.txt Compliance & Ethical Crawling

- **Alternative Approach:** We'll use a synthetic dataset generation approach that respects ethical guidelines while demonstrating all required techniques

Before scraping, we must respect robots.txt:- **Important:** Kayak blocks automated scrapers on booking pages for Terms of Service compliance

- Kayak's robots.txt is located at: https://booking.kayak.com/robots.txt- We check if our crawl paths are allowed before making requests

In [21]:
import requests
from urllib.robotparser import RobotFileParser
import re

print("=" * 60)
print("CHECKING ROBOTS.TXT COMPLIANCE")
print("=" * 60)

try:
    response = requests.get("https://booking.kayak.com/robots.txt", timeout=10)
    print("\n📄 Kayak's robots.txt content (first 1500 chars):")
    print("-" * 60)
    print(response.text[:1500])
    print("-" * 60)

    if "Disallow: /flights/" in response.text or "User-agent: *" in response.text:
        print("\n⚠️  FINDING: /flights/ paths are BLOCKED for automated scrapers")
        print("    Reason: Protects from price manipulation, server overload")
        print("\n✅ SOLUTION: Using synthetic realistic data generation")
        print("    (Maintains educational value while respecting ToS)")
    
except Exception as e:
    print(f"Could not fetch robots.txt: {e}")

CHECKING ROBOTS.TXT COMPLIANCE

📄 Kayak's robots.txt content (first 1500 chars):
------------------------------------------------------------
# robots.txt for production site.
#
# See http://www.robotstxt.org/wc/exclusion-admin.html
#
# default robots.txt for when a rewrite is not provided.
# Usually this is for internal tools

User-agent: *
Disallow: /
Noindex: /

------------------------------------------------------------

⚠️  FINDING: /flights/ paths are BLOCKED for automated scrapers
    Reason: Protects from price manipulation, server overload

✅ SOLUTION: Using synthetic realistic data generation
    (Maintains educational value while respecting ToS)
